In [1]:
!pip uninstall -y -q torchaudio torchcodec
!pip install -q vllm
!python -c "import torch, vllm; print('torch', torch.__version__, '| vllm', vllm.__version__, '| GPU', torch.cuda.device_count())"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/

In [2]:
import os, pandas as pd, numpy as np

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]

tr = pd.read_csv(f"{COMP}/train.csv")
allrep = (tr[["StudyInstanceUID", "Report"]].dropna(subset=["Report"])
          .rename(columns={"Report": "report"}))
allrep.to_csv("/kaggle/working/all_reports.csv", index=False)
lens = allrep["report"].astype(str).str.len()
print(f"Laporan: {len(allrep)} | median {int(lens.median())} karakter, maks {int(lens.max())}, "
      f"terpotong di 8000: {int((lens > 8000).sum())}")

Laporan: 4407 | median 977 karakter, maks 4743, terpotong di 8000: 0


In [3]:
%%writefile /kaggle/working/label_all.py
import os, re, json, argparse
import pandas as pd
from vllm import LLM, SamplingParams

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]
DEFS = {
    "ACL": "anterior cruciate ligament tear or injury (partial or complete, including tear of an ACL graft)",
    "MCL": "medial collateral ligament sprain, tear or injury",
    "Medial Meniscus": "tear of the medial meniscus",
    "Lateral Meniscus": "tear of the lateral meniscus",
    "Medial OA": "osteoarthritis or degenerative cartilage loss of the medial tibiofemoral compartment",
    "Lateral OA": "osteoarthritis or degenerative cartilage loss of the lateral tibiofemoral compartment",
    "PF OA": "osteoarthritis or degenerative cartilage loss of the patellofemoral compartment",
    "Effusion": "joint effusion",
    "Synovitis": "synovitis or synovial thickening/inflammation",
    "Baker's": "Baker's (popliteal) cyst",
    "Contusion": "bone contusion, bone bruise or traumatic bone marrow edema",
    "Fracture": "fracture of any bone of the knee (including avulsion or insufficiency fracture)",
}
PROMPT = """You are an expert musculoskeletal radiologist. Read the knee MRI report below. It may be written in any language.

For each finding, answer exactly one of "positive", "negative" or "uncertain":
- positive: the finding is clearly present, at any severity (including mild, small or partial).
- uncertain: the report says possible, suspected, cannot be excluded, or is equivocal.
- negative: the report states it is absent or normal, OR the finding is not mentioned.

Findings:
{defs}

Report:
\"\"\"
{report}
\"\"\"

Answer with a single JSON object only, using exactly these keys: {keys}"""

def to_num(v):
    v = str(v).strip().lower()
    if v.startswith("pos"): return 1.0
    if v.startswith("unc"): return 0.5
    if v.startswith("neg"): return 0.0
    return float("nan")

def parse(text):
    m = re.search(r"\{.*\}", text, re.S)
    out = {l: float("nan") for l in LABELS}
    if m:
        try:
            d = json.loads(m.group(0))
            for l in LABELS:
                if l in d: out[l] = to_num(d[l])
            return out
        except Exception:
            pass
    for l in LABELS:
        mm = re.search(re.escape(l) + r"\W+(positive|negative|uncertain)", text, re.I)
        if mm: out[l] = to_num(mm.group(1))
    return out

ap = argparse.ArgumentParser()
ap.add_argument("--model"); ap.add_argument("--out"); ap.add_argument("--quant", default=None)
ap.add_argument("--chunk", type=int, default=500)
a = ap.parse_args()

df = pd.read_csv("/kaggle/working/all_reports.csv")
done = pd.read_csv(a.out) if os.path.exists(a.out) else pd.DataFrame(columns=["StudyInstanceUID"])
todo = df[~df["StudyInstanceUID"].isin(done["StudyInstanceUID"])].reset_index(drop=True)
print(f"Sudah: {len(done)} | sisa: {len(todo)}", flush=True)

llm = LLM(model=a.model, quantization=a.quant, dtype="float16", tensor_parallel_size=2,
          max_model_len=6144, gpu_memory_utilization=0.90, enforce_eager=True,
          disable_custom_all_reduce=True)
tok = llm.get_tokenizer()
defs = "\n".join(f'- "{l}": {DEFS[l]}' for l in LABELS)
keys = ", ".join(f'"{l}"' for l in LABELS)
sp = SamplingParams(temperature=0, max_tokens=400)

for s in range(0, len(todo), a.chunk):
    part = todo.iloc[s:s + a.chunk]
    prompts = [tok.apply_chat_template(
                  [{"role": "user", "content": PROMPT.format(defs=defs, report=str(r)[:8000], keys=keys)}],
                  tokenize=False, add_generation_prompt=True) for r in part["report"]]
    outs = llm.generate(prompts, sp, use_tqdm=False)
    rows = [{"StudyInstanceUID": u, **parse(o.outputs[0].text), "raw": o.outputs[0].text}
            for u, o in zip(part["StudyInstanceUID"], outs)]
    done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True)
    done.to_csv(a.out, index=False)          # simpan setiap chunk
    print(f"  {len(done)}/{len(df)} laporan selesai", flush=True)

print(f"SELESAI | sel kosong: {int(done[LABELS].isna().sum().sum())}", flush=True)
os._exit(0)

Writing /kaggle/working/label_all.py


In [4]:
%%bash
cd /kaggle/working
echo "=== mulai $(date +%T)"
setsid timeout 10h python label_all.py --model Qwen/Qwen2.5-32B-Instruct-AWQ --quant awq --out labels_32b_all.csv > log_32b_all.txt 2>&1 &
pid=$!
wait $pid
echo "kode keluar: $?"
kill -9 -- -$pid 2>/dev/null
grep -E "Sudah|laporan selesai|SELESAI|Error|error" log_32b_all.txt | tail -n 15
echo "=== selesai $(date +%T)"

=== mulai 05:25:52
kode keluar: 0
Sudah: 0 | sisa: 4407
  500/4407 laporan selesai
  1000/4407 laporan selesai
  1500/4407 laporan selesai
  2000/4407 laporan selesai
  2500/4407 laporan selesai
  3000/4407 laporan selesai
  3500/4407 laporan selesai
  4000/4407 laporan selesai
  4407/4407 laporan selesai
SELESAI | sel kosong: 0
=== selesai 07:38:37


In [5]:
from sklearn.metrics import roc_auc_score

def find_file(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d != "competitions"]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)

def mean_auc(y, p):
    aucs = []
    for k in range(len(LABELS)):
        ok = ~np.isnan(y[:, k]) & (y[:, k] != 0.5) & ~np.isnan(p[:, k])
        aucs.append(roc_auc_score(y[ok, k], p[ok, k]) if len(np.unique(y[ok, k])) == 2 else np.nan)
    return np.array(aucs)

old = pd.read_csv(find_file("report_labels.csv")).set_index("StudyInstanceUID")[LABELS]
new = pd.read_csv("/kaggle/working/labels_32b_all.csv").set_index("StudyInstanceUID")[LABELS]
n_kosong = int(new.isna().sum().sum())
new = new.fillna(old.reindex(new.index))                  # cadangan: label lama untuk sel kosong
print(f"Studi: {len(new)} | sel kosong dari 32B: {n_kosong} | setelah diisi label lama: {int(new.isna().sum().sum())}")

out = new.reset_index()
out.to_csv("/kaggle/working/report_labels.csv", index=False)  # nama sama, supaya notebook lain tidak perlu diubah

gold = tr[tr[LABELS].notna().all(axis=1)].set_index("StudyInstanceUID")[LABELS]
Y = gold.values.astype(float)
for name, P in [("lama 7B", old), ("32B", new), ("rata-rata lama + 32B", (old.reindex(new.index) + new) / 2)]:
    a = mean_auc(Y, P.reindex(gold.index).values.astype(float))
    print(f"{name:22s} mean AUC gold: {np.nanmean(a):.4f}")

print("\nDistribusi label baru (proporsi positif):")
print((out[LABELS] == 1).mean().round(3).to_string())

Studi: 4407 | sel kosong dari 32B: 0 | setelah diisi label lama: 0
lama 7B                mean AUC gold: 0.8092
32B                    mean AUC gold: 0.8335
rata-rata lama + 32B   mean AUC gold: 0.8465

Distribusi label baru (proporsi positif):
ACL                 0.194
MCL                 0.123
Medial Meniscus     0.448
Lateral Meniscus    0.184
Medial OA           0.320
Lateral OA          0.202
PF OA               0.390
Effusion            0.600
Synovitis           0.215
Baker's             0.266
Contusion           0.299
Fracture            0.045
